# YOLO + DeepSort演示

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# 初始化 YOLO 模型和 DeepSORT 跟踪器
yolo_model = YOLO("yolov8l.pt")
tracker = DeepSort(max_age=10, n_init=5, nn_budget=50, max_iou_distance=0.9)

# 打开视频文件
video_path = "LifeguardRescueVideos/video_2.mp4"
cap = cv2.VideoCapture(video_path)

frame_count = 0
skip_frames = 10  # 每隔 10 帧处理一次

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % skip_frames != 0: 
        continue

    h, w, _ = frame.shape

    # 使用 YOLOv8 进行目标检测
    results = yolo_model(frame, conf=0.1, iou=0.4, batch=16)
    detections = []

    for r in results:
        for box in r.boxes:
            if int(box.cls) == 0:  # 仅处理 "person"
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = float(box.conf[0])

                # 调整边界框范围
                padding = 20
                x1 = max(0, x1 - padding)
                y1 = max(0, y1 - padding)
                x2 = min(w, x2 + padding)
                y2 = min(h, y2 + padding)

                # 添加到跟踪检测输入
                detections.append(([x1, y1, x2 - x1, y2 - y1], confidence, "person"))

    # 更新跟踪信息
    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed(): 
            continue

        # 获取边界框并调整范围
        x1, y1, x2, y2 = map(int, track.to_ltrb())
        padding = 20
        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(w, x2 + padding)
        y2 = min(h, y2 + padding)

        # 绘制跟踪框和 ID
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {track.track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # 显示结果（按 "q" 键退出）
    cv2.imshow("YOLO + DeepSORT", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# 释放资源
cap.release()
cv2.destroyAllWindows()

# 将原视频按不同track_id分别生成图片

In [1]:
import cv2
import os
import torch
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

# ========================
# 1. 初始化 YOLO 模型
# ========================
yolo_model = YOLO("yolov8l.pt")

# 检查是否在 GPU 上
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ YOLOv8 正在使用: {device.upper()}")

# ========================
# 2. 输入/输出目录
# ========================
video_dir = "LifeguardRescueVideos"  # 读取视频所在目录
output_dir = "tracked_images"        # 存储检测结果
os.makedirs(output_dir, exist_ok=True)


def process_video(video_file):
    video_path = os.path.join(video_dir, video_file)
    video_name = os.path.splitext(video_file)[0]  
    video_output_dir = os.path.join(output_dir, video_name) 
    os.makedirs(video_output_dir, exist_ok=True)

    print(f"📹 正在处理视频: {video_file} ...")

    # ========================
    # 3. 初始化 DeepSORT
    # ========================
    tracker = DeepSort(
        max_age=10,
        n_init=5,
        nn_budget=50,
        max_iou_distance=0.9
    )

    # 打开视频
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    skip_frames = 10  # 每隔 10 帧处理一次

    # 获取视频 FPS、分辨率、总帧数
    fps = int(round(cap.get(cv2.CAP_PROP_FPS)))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    padding = 20

    # ========================
    # 4. 存储裁剪的行人图像
    # ========================
    collected_crops = defaultdict(list)

    # ========================
    # 5. 循环读取视频帧并检测
    # ========================
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        if frame_count % skip_frames != 0:
            continue

        current_time = frame_count / fps
        print(f"📸 正在处理 {video_name} - 帧 {frame_count}/{total_frames} ({current_time:.2f}s)...")
        results = yolo_model(frame, conf=0.1, iou=0.4, batch=16, verbose=False)
        fh, fw, _ = frame.shape
        detections = []

        # 处理 YOLO 检测结果
        for r in results:
            for box in r.boxes:
                if int(box.cls) == 0: #仅处理类别为 person
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    confidence = float(box.conf[0])

                    x1 = max(0, x1 - padding)
                    y1 = max(0, y1 - padding)
                    x2 = min(fw, x2 + padding)
                    y2 = min(fh, y2 + padding)

                    if x2 > x1 and y2 > y1:
                        w_box = x2 - x1
                        h_box = y2 - y1
                        detections.append(([x1, y1, w_box, h_box], confidence, "person"))

        # ========================
        # 6. 更新 DeepSORT 跟踪信息
        # ========================
        tracks = tracker.update_tracks(detections, frame=frame)

        # ========================
        # 7. 对每个已确认的轨迹进行处理
        # ========================
        for track in tracks:
            if not track.is_confirmed():
                continue

            # 获取检测框的左上、右下坐标 (ltrb)
            l, t, r, b = map(int, track.to_ltrb())  # left, top, right, bottom

            xx1 = max(0, l - padding)
            yy1 = max(0, t - padding)
            xx2 = min(fw, r + padding)
            yy2 = min(fh, b + padding)

            # 确保不越界且有有效面积
            if xx2 > xx1 and yy2 > yy1:
                person_crop = frame[yy1:yy2, xx1:xx2]
            else:
                continue

            if person_crop.size > 0:
                # 获取 track id (本视频内唯一)
                track_id = track.track_id

                # 计算一些信息（如目标中心点）
                w_box = xx2 - xx1
                h_box = yy2 - yy1
                center_x = xx1 + w_box // 2
                center_y = yy1 + h_box // 2

                # 生成文件名
                image_name = (f"time_{current_time:.2f}s_{center_x}_{center_y}_"
                              f"{w_box}x{h_box}_{frame_width}x{frame_height}.png")

                collected_crops[track_id].append((image_name, person_crop))

    cap.release()

    # ========================
    # 8. 统一写文件
    # ========================
    print(f"🔄 正在将 {video_name} 中所有裁剪图像一次性写入文件...")
    for track_id, crops in collected_crops.items():
        person_output_dir = os.path.join(video_output_dir, f"person_{track_id}")
        os.makedirs(person_output_dir, exist_ok=True)

        for image_name, crop in crops:
            image_path = os.path.join(person_output_dir, image_name)
            cv2.imwrite(image_path, crop)

    print(f"✅ 视频 {video_file} 处理完成，并已一次性写出所有裁剪图像！")


# ========================
# 9. 并行处理视频入口
# ========================
if __name__ == "__main__":
    # 获取所有视频文件
    video_files = [
        f for f in os.listdir(video_dir)
        if f.endswith(('.mp4', '.avi', '.mov'))
    ]

    with ThreadPoolExecutor(max_workers=8) as executor:
        executor.map(process_video, video_files)

    print("✅ 所有视频处理完成！")


✅ YOLOv8 正在使用: CPU
📹 正在处理视频: video_101.mp4 ...
📹 正在处理视频: video_102.mp4 ...
📸 正在处理 video_101 - 帧 10/1059 (0.33s)...
📸 正在处理 video_102 - 帧 10/2049 (0.33s)...
Ultralytics 8.3.73  Python-3.9.21 torch-2.6.0+cpu CPU (AMD Ryzen 7 7840HS with Radeon 780M Graphics)
YOLOv8l summary (fused): 268 layers, 43,668,288 parameters, 0 gradients, 165.2 GFLOPs
📸 正在处理 video_101 - 帧 20/1059 (0.67s)...
📸 正在处理 video_102 - 帧 20/2049 (0.67s)...
📸 正在处理 video_101 - 帧 30/1059 (1.00s)...
📸 正在处理 video_102 - 帧 30/2049 (1.00s)...
📸 正在处理 video_101 - 帧 40/1059 (1.33s)...
📸 正在处理 video_102 - 帧 40/2049 (1.33s)...
📸 正在处理 video_101 - 帧 50/1059 (1.67s)...
📸 正在处理 video_102 - 帧 50/2049 (1.67s)...
📸 正在处理 video_101 - 帧 60/1059 (2.00s)...
📸 正在处理 video_102 - 帧 60/2049 (2.00s)...
📸 正在处理 video_101 - 帧 70/1059 (2.33s)...
📸 正在处理 video_102 - 帧 70/2049 (2.33s)...
📸 正在处理 video_101 - 帧 80/1059 (2.67s)...
📸 正在处理 video_102 - 帧 80/2049 (2.67s)...
📸 正在处理 video_101 - 帧 90/1059 (3.00s)...
📸 正在处理 video_102 - 帧 90/2049 (3.00s)...
📸 正在处理 video_101 -

📸 正在处理 video_102 - 帧 940/2049 (31.33s)...
📸 正在处理 video_101 - 帧 950/1059 (31.67s)...
📸 正在处理 video_102 - 帧 950/2049 (31.67s)...
📸 正在处理 video_101 - 帧 960/1059 (32.00s)...
📸 正在处理 video_102 - 帧 960/2049 (32.00s)...
📸 正在处理 video_101 - 帧 970/1059 (32.33s)...
📸 正在处理 video_102 - 帧 970/2049 (32.33s)...
📸 正在处理 video_101 - 帧 980/1059 (32.67s)...
📸 正在处理 video_102 - 帧 980/2049 (32.67s)...
📸 正在处理 video_101 - 帧 990/1059 (33.00s)...
📸 正在处理 video_102 - 帧 990/2049 (33.00s)...
📸 正在处理 video_101 - 帧 1000/1059 (33.33s)...
📸 正在处理 video_102 - 帧 1000/2049 (33.33s)...
📸 正在处理 video_101 - 帧 1010/1059 (33.67s)...
📸 正在处理 video_102 - 帧 1010/2049 (33.67s)...
📸 正在处理 video_101 - 帧 1020/1059 (34.00s)...
📸 正在处理 video_102 - 帧 1020/2049 (34.00s)...
📸 正在处理 video_101 - 帧 1030/1059 (34.33s)...
📸 正在处理 video_102 - 帧 1030/2049 (34.33s)...
📸 正在处理 video_101 - 帧 1040/1059 (34.67s)...
📸 正在处理 video_102 - 帧 1040/2049 (34.67s)...
📸 正在处理 video_101 - 帧 1050/1059 (35.00s)...
📸 正在处理 video_102 - 帧 1050/2049 (35.00s)...
🔄 正在将 video_101 中所有裁剪图

# 将原视频处理成带track_id的视频

In [ ]:
import cv2
import os
import torch
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from concurrent.futures import ThreadPoolExecutor

# ========================
# 1. 初始化 YOLO 模型
# ========================
yolo_model = YOLO("yolov8l.pt")

# 检查是否在 GPU 上
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ YOLOv8 正在使用: {device.upper()}")

# ========================
# 2. 输入/输出目录
# ========================
video_dir = "LifeguardRescueVideos"    
output_dir = "tracked_videos"         
os.makedirs(output_dir, exist_ok=True)

def process_video(video_file):
    video_path = os.path.join(video_dir, video_file)
    video_name = os.path.splitext(video_file)[0] 

    print(f"📹 正在处理视频: {video_file} ...")

    # ========================
    # 3. 初始化 DeepSORT
    # ========================
    tracker = DeepSort(
        max_age=10,
        n_init=5,
        nn_budget=50,
        max_iou_distance=0.9
    )

    # 打开视频
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    skip_frames = 10  
    
    fps = int(round(cap.get(cv2.CAP_PROP_FPS)))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    padding = 20

    # ========================
    # 4. 初始化“简短视频”的 VideoWriter
    # ========================
    short_video_name = f"{video_name}_short.mp4"
    short_video_path = os.path.join(output_dir, short_video_name)

    new_fps = max(1, fps // skip_frames)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    short_video_writer = cv2.VideoWriter(
        short_video_path,
        fourcc,
        new_fps,
        (frame_width, frame_height)
    )

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        if frame_count % skip_frames != 0:
            continue

        current_time = frame_count / fps
        print(f"📸 正在处理 {video_name} - 帧 {frame_count}/{total_frames} ({current_time:.2f}s)...")

        # ================
        # YOLOv8 目标检测
        # ================
        results = yolo_model(frame, conf=0.1, iou=0.4, batch=16, verbose=False)
        fh, fw, _ = frame.shape
        detections = []

        # 处理 YOLO 检测结果
        for r in results:
            for box in r.boxes:
                if int(box.cls) == 0: #仅处理类别为 person
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    confidence = float(box.conf[0])

                    x1 = max(0, x1 - padding)
                    y1 = max(0, y1 - padding)
                    x2 = min(fw, x2 + padding)
                    y2 = min(fh, y2 + padding)

                    if x2 > x1 and y2 > y1:
                        w_box = x2 - x1
                        h_box = y2 - y1
                        detections.append(([x1, y1, w_box, h_box], confidence, "person"))

        # ================
        # 更新 DeepSORT 跟踪信息
        # ================
        tracks = tracker.update_tracks(detections, frame=frame)

        # ================
        # 遍历已确认轨迹，在帧上画框和 Track ID
        # ================
        for track in tracks:
            if not track.is_confirmed():
                continue

            l, t, r, b = map(int, track.to_ltrb())  # left, top, right, bottom

            xx1 = max(0, l - padding)
            yy1 = max(0, t - padding)
            xx2 = min(fw, r + padding)
            yy2 = min(fh, b + padding)

            # 在当前帧上绘制
            color = (0, 255, 0)  # 绿色
            thickness = 2
            cv2.rectangle(frame, (xx1, yy1), (xx2, yy2), color, thickness)
            cv2.putText(
                frame,
                f"ID {track.track_id}",
                (xx1, yy1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

        # ================
        # 将带标注的帧写进简短视频
        # ================
        short_video_writer.write(frame)

    # 结束后释放资源
    cap.release()
    short_video_writer.release()

    print(f"✅ 视频 {video_file} 处理完成！简短视频已输出: {short_video_path}")


if __name__ == "__main__":
    # 获取所有视频文件
    video_files = [
        f for f in os.listdir(video_dir)
        if f.endswith(('.mp4', '.avi', '.mov'))
    ]
    
    with ThreadPoolExecutor(max_workers=8) as executor:
        executor.map(process_video, video_files)

    print("✅ 所有视频处理完成！")
